# Intersect Community Data Workflow for Person Record Files with Disability

# This notebook includes Person Record File with optional Disability Status

## Overview
This code works with the the person record file (PREC) workflow and adds disability status.


## Required Inputs
Program requires the following inputs:

[Census API KEY *REQUIRED*](CENSUS_API_KEY.md) See CENSUS_API_KEY.md file for more details.
    
## Output Description
The output of this workflow is a CSV file with the person record file with disability status.

## Instructions
Users can run the workflow by executing each block of code in the notebook.

## Description of Program
- program:    ncoda_07kv1_PREC_Disability
- task:       run PREC with disability status
- See github commits for description of program updates
- Current Version: v1 - 
- 2026-09-01 - Integrate disability work into PREC workflow
- project:    Texas Mitigation Planning Initiative
- funding:	  FEMA
- author:     Nathanael Rosenheim and Swastika Barua

## Required Citations:
Rosenheim, Nathanael, Roberto Guidotti, Paolo Gardoni & Walter Gillis Peacock. (2021). Integration of detailed household and housing unit characteristic data with critical infrastructure for post-hazard resilience modeling. _Sustainable and Resilient Infrastructure_. 6(6), 385-401. https://doi.org/10.1080/23789689.2019.1681821

Rosenheim, Nathanael (2021) “Detailed Household and Housing Unit Characteristics: Data and Replication Code.” _DesignSafe-CI_. 
https://doi.org/10.17603/ds2-jwf6-s535.

## Loading the Community

In [1]:
# To reload submodules need to use this magic command to set autoreload on
%load_ext autoreload
%autoreload 2
from pyncoda.ncoda_00g_community_options import *
from IPython.display import display

In [2]:
# select a community from this list
# if your community is not in this list, add it to the file ncoda_00g_community_options.py
list_community_options(communities_dictionary)

['Lumberton, NC: IN-CORE Building inventory for Robeson County, NC',
 'Galveston, TX: IN-CORE Building inventory for Galveston County, TX',
 'Galveston, TX: NSI Building inventory for Galveston County, TX',
 'Galveston, TX: IN-CORE Building inventory for Galveston Island, TX',
 'Mayfield, KY: NSI Building inventory for Graves County, KY',
 'Beaumont, TX: NSI Building inventory for Jefferson County, TX',
 'Beaumont, TX: Safayet Building inventory for Jefferson County, TX',
 'Pentwater, MI: NSI Building inventory for Oceana County, MI',
 'Seaside, OR: NSI Building inventory for Clatsop County, OR',
 'Lane County, OR: NSI Building inventory for Lane County, OR',
 'Benton County, OR: NSI Building inventory for Benton County, OR',
 'Southeast Texas Urban Integrated Field Lab: NSI Building inventory for Southeast Texas',
 'Southeast Texas Urban Integrated Field Lab (12 neighbor counties): NSI Building inventory for Southeast Texas',
 'Brazos County, TX: NSI Building inventory for Brazos Coun

In [3]:
community_id_by_name = 'Seaside, OR: NSI Building inventory for Clatsop County, OR'

In [4]:
community_id, focalplace, countyname, countyfips = get_community_id_by_name(community_id_by_name)
communities = {community_id : communities_dictionary[community_id]}

Selected community ID: Seaside_OR_NSI
Seaside, OR is in OREGON
Focal place: Seaside
Seaside, OR is in Clatsop County, OR with FIPS code 41007
Use IN-CORE: False


## Setiing up the Environment

In [5]:
import pandas as pd
import geopandas as gpd # For reading in shapefiles
import numpy as np
import sys # For displaying package versions
import os # For managing directories and file paths if drive is mounted
import scooby # Reports Python environment

import contextily as cx # For adding basemap tiles to plot
import matplotlib.pyplot as plt # For plotting and making graphs

In [6]:
# open, read, and execute python program with reusable commands
from pyncoda.ncoda_00d_cleanvarsutils import *
from pyncoda.ncoda_04c_poptableresults import *
from pyncoda.ncoda_07i_process_communities import process_community_workflow

In [7]:
# Generate report of Python environment
base_packages = ['pandas','ipyleaflet','seaborn','contextily']
incore_packages = ['pyincore','pyincore_viz']
check_packages = base_packages + incore_packages
print(scooby.Report(additional=check_packages))


--------------------------------------------------------------------------------
  Date: Tue Sep 01 17:09:40 2026 Eastern Daylight Time

                OS : Windows (10 10.0.26200 SP0 Multiprocessor Free)
            CPU(s) : 16
           Machine : AMD64
      Architecture : 64bit
               RAM : 31.7 GiB
       Environment : Jupyter

  Python 3.10.14 | packaged by Anaconda, Inc. | (main, May  6 2024, 19:44:50)
  [MSC v.1916 64 bit (AMD64)]

            pandas : 2.2.2
        ipyleaflet : Module not found
           seaborn : 0.13.2
        contextily : 1.6.0
          pyincore : Module not found
      pyincore_viz : Module not found
             numpy : 1.26.4
             scipy : 1.13.1
           IPython : 8.25.0
        matplotlib : 3.8.4
            scooby : 0.10.0

  Intel(R) oneAPI Math Kernel Library Version 2023.1-Product Build 20230303
  for Intel(R) 64 architecture applications
--------------------------------------------------------------------------------


In [8]:
# Check working directory - good practice for relative path access
os.getcwd()

'c:\\Users\\nathanael99\\MyProjects\\GitHub\\intersect-community-data'

In [9]:
basevintage_options = ['2010']
base_seed = 9876
iterations = 1

In [10]:
from pyncoda.ncoda_07i_process_communities import process_community_workflow

PREC Workflow (Person Records with Disability)

In [11]:
# ==============================================================
# Step 1: Setup — imports and community configuration
# ==============================================================

%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
from IPython.display import display

from pyncoda.ncoda_00g_community_options import *
from pyncoda.CommunitySourceData.api_census_gov.acg_01a_BaseInventory import BaseInventory
from pyncoda.CommunitySourceData.api_census_gov.acg_00f_preci_block2010 import sexbyage_P12_varstem_roots
from pyncoda.CommunitySourceData.api_census_gov.acg_00a_createAPI_datastructure import createAPI_datastructure

# --- Community Setup ---
community_id_by_name = 'Seaside, OR: NSI Building inventory for Clatsop County, OR'
community_id, focalplace, countyname, countyfips = get_community_id_by_name(community_id_by_name)

state_county      = countyfips
state_county_name = countyname
seed              = 9876
basevintage       = '2010'

# --- Output folders ---
prec_outputfolder = "OutputData"
prec_base = f"{prec_outputfolder}/{community_id}"

prec_outputfolders = {
    'top'                    : prec_base,
    'logfiles'               : f"{prec_base}/00_logfiles",
    'CommunitySourceData'    : f"{prec_base}/01_CommunitySourceData",
    'TidyCommunitySourceData': f"{prec_base}/02_TidyCommunitySourceData",
    'BaseInventory'          : f"{prec_base}/03_BaseInventory",
    'RandomMerge'            : f"{prec_base}/04_RandomMerge",
}
for folder in prec_outputfolders.values():
    os.makedirs(folder, exist_ok=True)

print(f"Community    : {community_id}")
print(f"State/County : {state_county} — {state_county_name}")
print(f"Seed         : {seed}")
print(f"Base vintage : {basevintage}")
print("Folders ready ✓")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Selected community ID: Seaside_OR_NSI
Seaside, OR is in OREGON
Focal place: Seaside
Seaside, OR is in Clatsop County, OR with FIPS code 41007
Use IN-CORE: False
Community    : Seaside_OR_NSI
State/County : 41007 — Clatsop County, OR
Seed         : 9876
Base vintage : 2010
Folders ready ✓


In [12]:
# ==============================================================
# Step 2: Create block-level person records from P12 (Sex by Age)
# ==============================================================
block_df = {}

block_df['preci'] = BaseInventory.get_apidata(
    state_county  = state_county,
    geo_level     = 'block',
    vintage       = basevintage,
    mutually_exclusive_varstems_roots_dictionaries = [sexbyage_P12_varstem_roots],
    outputfolders = prec_outputfolders,
    outputfile    = 'CorePREC'
)

print(f"\nStep 2 complete: Block-level person records created.")
print(f"Shape   : {block_df['preci'].shape}")
print(f"Columns : {list(block_df['preci'].columns)}")
display(block_df['preci'].head(3))

{'top': 'OutputData/Seaside_OR_NSI', 'logfiles': 'OutputData/Seaside_OR_NSI/00_logfiles', 'CommunitySourceData': 'OutputData/Seaside_OR_NSI/01_CommunitySourceData', 'TidyCommunitySourceData': 'OutputData/Seaside_OR_NSI/02_TidyCommunitySourceData', 'BaseInventory': 'OutputData/Seaside_OR_NSI/03_BaseInventory', 'RandomMerge': 'OutputData/Seaside_OR_NSI/04_RandomMerge'}
File OutputData/Seaside_OR_NSI/02_TidyCommunitySourceData/P12IA-G_41007_2010.csv Already exists - Skipping API Call.


c:\Users\nathanael99\MyProjects\GitHub\intersect-community-data\pyncoda\CommunitySourceData\api_census_gov\acg_01a_BaseInventory.py:453: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['41' '41' '41' ... '41' '41' '41']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:,geo_level] =  df[geo_level].apply(lambda x: str(x).zfill(len))
c:\Users\nathanael99\MyProjects\GitHub\intersect-community-data\pyncoda\CommunitySourceData\api_census_gov\acg_01a_BaseInventory.py:453: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['007' '007' '007' ... '007' '007' '007']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:,geo_level] =  df[geo_level].apply(lambda x: str(x).zfill(len))
c:\Users\nathanael99\MyProjects\GitHub\intersect-community-data\pyncoda\CommunitySourceD


Step 2 complete: Block-level person records created.
Shape   : (37039, 9)
Columns : ['precid', 'Block2010', 'Block2010str', 'sex', 'minageyrs', 'maxageyrs', 'race', 'hispan', 'prec_counter']


,precid,Block2010,Block2010str,sex,minageyrs,maxageyrs,race,hispan,prec_counter
0,B410079501001001P001,410079501001001,B410079501001001,1,21,21,1,0,1
1,B410079501001001P002,410079501001001,B410079501001001,1,55,59,1,0,2
2,B410079501001001P003,410079501001001,B410079501001001,1,67,69,1,0,3


In [13]:
# ==============================================================
# Step 3: Add Hispanic ethnicity via graft method
# ==============================================================

from pyncoda.CommunitySourceData.api_census_gov.acg_00f_preci_block2010 import (
    sexbyage_P12HAI_varstem_roots,
    hispan_byrace_P5_varstem_roots
)

block_df['precihispan'] = BaseInventory.graft_on_new_char(
    base_inventory        = block_df['preci'],
    state_county          = state_county,
    new_char              = 'hispan',
    new_char_dictionaries = [
        sexbyage_P12HAI_varstem_roots,
        hispan_byrace_P5_varstem_roots
    ],
    outputfile    = 'preci',
    outputfolders = prec_outputfolders
)

print(f"\nStep 3 complete: Hispanic ethnicity added.")
print(f"Shape   : {block_df['precihispan'].shape}")
print(f"Columns : {list(block_df['precihispan'].columns)}")
print(f"\nhispan value counts:")
print(block_df['precihispan']['hispan'].value_counts().sort_index())
display(block_df['precihispan'].head(3))

Base Housing Unit Inventory has new characteristic hispan
Graft process will predict missing values of  hispan

***************************************
    Base Inventory has 3359 observations not set
***************************************


***************************************
    Predicting hispan based on ['sex', 'minageyrs', 'maxageyrs', 'hispanbyP12HAI', 'byracehispan'] ['sex', 'minageyrs', 'maxageyrs'] P12HAI
***************************************


***************************************
    Base Inventory has 3359 observations not set
***************************************

File OutputData/Seaside_OR_NSI/02_TidyCommunitySourceData/P12HAI_41007_2010.csv Already exists - Skipping API Call.
hispanbyP12HAI
Fill missing values with 0

***************************************
    Predicting hispan based on ['race', 'hispanbyP5'] ['race'] P5
***************************************


***************************************
    Base Inventory has 617 observations not set
**********

c:\Users\nathanael99\MyProjects\GitHub\intersect-community-data\pyncoda\CommunitySourceData\api_census_gov\acg_01a_BaseInventory.py:453: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['41' '41' '41' ... '41' '41' '41']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:,geo_level] =  df[geo_level].apply(lambda x: str(x).zfill(len))
c:\Users\nathanael99\MyProjects\GitHub\intersect-community-data\pyncoda\CommunitySourceData\api_census_gov\acg_01a_BaseInventory.py:453: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['007' '007' '007' ... '007' '007' '007']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:,geo_level] =  df[geo_level].apply(lambda x: str(x).zfill(len))
c:\Users\nathanael99\MyProjects\GitHub\intersect-community-data\pyncoda\CommunitySourceD

Fill missing values with 0
['Block2010str', 'sex', 'minageyrs', 'maxageyrs']
Updating hispanbyP12HAI_counter  based on total probability and ['Block2010str', 'sex', 'minageyrs', 'maxageyrs']
['Block2010str', 'race']
Updating hispanbyP5_counter  based on total probability and ['Block2010str', 'race']
Merge vars for hispanbyP12HAI_counter = ['Block2010str', 'sex', 'minageyrs', 'maxageyrs']
Updating flags based on preccount_hispanbyP12HAIupdated
Merge vars for hispanbyP5_counter = ['Block2010str', 'race']
Updating flags based on preccount_hispanbyP5updated
Length of Set1 split dataframe 33680
Length of Not Set split dataframe 141
Length of hispanbyP12HAI split dataframe 2742
Length of hispanbyP5 split dataframe 316
Length of hispanbyP12HAI_counter split dataframe 153
Length of hispanbyP5_counter split dataframe 7


Shape of dataframe after total sum: (37039, 24)
['prob_hispanbyP12HAI', 'prob_hispanbyP5']
hispanbyP12HAI
hispanbyP5

Step 3 complete: Hispanic ethnicity added.
Shape   : (3703

,precid,Block2010,Block2010str,sex,minageyrs,maxageyrs,race,hispan,prec_counter,hispan_flag,...,sumby_hispanbyP12HAI,prob_hispanbyP12HAI,hispanbyP5,preccount_hispanbyP5,sumby_hispanbyP5,prob_hispanbyP5,hispanbyP12HAI_counter,hispanbyP5_counter,preccount_hispanbyP12HAIupdated,preccount_hispanbyP5updated
0,B410079501001001P001,410079501001001,B410079501001001,1.0,21.0,21.0,1.0,0.0,1.0,hispan set to 0 by core hui,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B410079501001001P002,410079501001001,B410079501001001,1.0,55.0,59.0,1.0,0.0,2.0,hispan set to 0 by core hui,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,B410079501001001P003,410079501001001,B410079501001001,1.0,67.0,69.0,1.0,0.0,3.0,hispan set to 0 by core hui,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# ==============================================================
# Step 4: Add random age to block and tract records
# ==============================================================

from pyncoda.CommunitySourceData.api_census_gov.acg_02c_agefunctions import (
    add_randage,
    add_P12age_groups
)

# --- Step 4a: Add random age to block-level person records ---
block_df['precihispan'] = add_randage(
    block_df['precihispan'],
    seed    = seed,
    varname = 'randageP12'
)
block_df['precihispan'] = add_P12age_groups(
    block_df['precihispan'],
    varname = 'randageP12'
)

print(f"Step 4a complete: Random age added to block records.")
print(f"Shape      : {block_df['precihispan'].shape}")
print(f"\nrandageP12 sample stats:")
print(block_df['precihispan']['randageP12'].describe())
print(f"\nagegroupP12 value counts:")
print(block_df['precihispan']['agegroupP12'].value_counts().sort_index())

Step 4a complete: Random age added to block records.
Shape      : (37039, 26)

randageP12 sample stats:
count    37039.00000
mean        40.71233
std         23.68020
min          0.00000
25%         21.00000
50%         42.00000
75%         58.00000
max        109.00000
Name: randageP12, dtype: float64

agegroupP12 value counts:
agegroupP12
1.0     2076
2.0     2034
3.0     2111
4.0     1380
5.0      998
6.0      484
7.0      455
8.0     1352
9.0     2207
10.0    2044
11.0    2017
12.0    2068
13.0    2602
14.0    2892
15.0    3291
16.0    1240
17.0    1628
18.0     910
19.0    1167
20.0    1393
21.0    1043
22.0     850
23.0     797
Name: count, dtype: int64


In [15]:
# ==============================================================
# Step 5: Fetch tract-level PCT12 data and add random age
# ==============================================================

tract_df = {}
group = 'PCT12'

# --- Step 5a: Fetch PCT12 variable structure ---
sexbyage_PCT12_varstem_roots = createAPI_datastructure.obtain_api_metadata(
    vintage      = '2010',
    dataset_name = 'dec/sf1',
    group        = group,
    outputfolder = prec_outputfolder
)

# --- Step 5b: Pull tract-level PCT12 data ---
tract_df['PCT12'] = BaseInventory.get_apidata(
    state_county  = state_county,
    geo_level     = 'tract',
    vintage       = '2010',
    mutually_exclusive_varstems_roots_dictionaries = [sexbyage_PCT12_varstem_roots],
    outputfolders = prec_outputfolders,
    outputfile    = group
)

# --- Step 5c: Add random age to tract records ---
tract_df['PCT12'] = add_randage(
    tract_df['PCT12'],
    seed    = seed,
    varname = 'randagePCT12'
)
tract_df['PCT12'] = add_P12age_groups(
    tract_df['PCT12'],
    varname = 'randagePCT12'
)

print(f"Step 5 complete: Tract-level PCT12 data fetched and age added.")
print(f"Shape   : {tract_df['PCT12'].shape}")
print(f"Columns : {list(tract_df['PCT12'].columns)}")
print(f"\nrandagePCT12 sample stats:")
print(tract_df['PCT12']['randagePCT12'].describe())
display(tract_df['PCT12'].head(3))

Dictionary file OutputData/00_datastructures/api_census_gov/PCT12_2010_datastructurev0-2-0.txt Already exists - Skipping API Call.
{'top': 'OutputData/Seaside_OR_NSI', 'logfiles': 'OutputData/Seaside_OR_NSI/00_logfiles', 'CommunitySourceData': 'OutputData/Seaside_OR_NSI/01_CommunitySourceData', 'TidyCommunitySourceData': 'OutputData/Seaside_OR_NSI/02_TidyCommunitySourceData', 'BaseInventory': 'OutputData/Seaside_OR_NSI/03_BaseInventory', 'RandomMerge': 'OutputData/Seaside_OR_NSI/04_RandomMerge'}
File OutputData/Seaside_OR_NSI/02_TidyCommunitySourceData/PCT12_41007_2010.csv Already exists - Skipping API Call.
Step 5 complete: Tract-level PCT12 data fetched and age added.
Shape   : (37039, 11)
Columns : ['index', 'GEO_ID', 'state', 'county', 'tract', 'sex', 'minageyrs', 'maxageyrs', 'prec_counter', 'randagePCT12', 'agegroupP12']

randagePCT12 sample stats:
count    37039.000000
mean        41.012770
std         23.252288
min          0.000000
25%         21.000000
50%         43.000000
7

,index,GEO_ID,state,county,tract,sex,minageyrs,maxageyrs,prec_counter,randagePCT12,agegroupP12
0,0,1400000US41007950100,41,7,950100,1,0,0,1,0.0,1.0
1,0,1400000US41007950100,41,7,950100,1,0,0,2,0.0,1.0
2,0,1400000US41007950100,41,7,950100,1,0,0,3,0.0,1.0


In [16]:
# ==============================================================
# Step 6a: Random merge tract age onto block person records
# ==============================================================

from pyncoda.CommunitySourceData.api_census_gov.acg_02a_add_categorical_char import add_new_char_by_random_merge_2dfs

add_age = add_new_char_by_random_merge_2dfs(
    dfs = {
        'primary'  : {'data'       : block_df['precihispan'],
                      'primarykey' : 'precid',
                      'geolevel'   : 'Block',
                      'geovintage' : '2010',
                      'notes'      : 'Block age, sex, race, ethnicity data.'},
        'secondary': {'data'       : tract_df['PCT12'],
                      'primarykey' : 'uniqueidPCT12',
                      'geolevel'   : 'Tract',
                      'geovintage' : '2010',
                      'notes'      : 'Tract single-year age, sex data.'}
    },
    seed              = seed,
    common_group_vars = ['agegroupP12'],
    new_char          = 'randagePCT12',
    geolevel          = 'Tract',
    geovintage        = '2010',
    by_groups         = {'All': {'by_variables': ['sex']}},
    fillna_value      = -999,
    state_county      = state_county,
    outputfile        = 'preci_randomage',
    outputfolder      = prec_outputfolders['RandomMerge']
)

rounds = {'options': {
    'option1': {
        'notes'            : 'By original common group vars and by groups variables.',
        'common_group_vars': add_age.common_group_vars,
        'by_groups'        : add_age.by_groups
    }},
    'geo_levels': ['Tract']
}

prec_age_df = add_age.run_random_merge_2dfs(rounds)

# Sanity check
primary_df = prec_age_df['primary']
print(f"\nStep 6 complete: Tract age merged onto block person records.")
print(f"Shape   : {primary_df.shape}")
print(f"Columns : {list(primary_df.columns)}")
print(f"\nrandagePCT12 assignment flags:")
print(primary_df['randagePCT12_flagsetrm'].value_counts().sort_index())
print(f"\nrandagePCT12 stats:")
print(primary_df['randagePCT12'].describe())
display(primary_df.head(3))

Round 1
Performing random merge at geography level: Tract
By original common group vars and by groups variables.
Running random merge by ['Tract2010', 'sex', 'agegroupP12']
    Setting up  primary data with primary key and flags
Geolevels available []
Geolvarids available ['Block2010']
Adding Tract2010 expected length 11
Dataframe has Block 2010 for new geovar Tract2010
Confirming Tract2010 has expected length.
Checking primary key name precid
Primary key variable precid is unique.
Primary key precid has no missing values
Initializing primary flag set variable for randagePCT12
New flag: randagePCT12_flagsetrm
['precid', 'Tract2010', 'Block2010', 'Block2010str', 'sex', 'minageyrs', 'maxageyrs', 'race', 'hispan', 'prec_counter', 'hispan_flag', 'hispan_flagset', 'totalprob_hispan', 'hispanbyP12HAI', 'preccount_hispanbyP12HAI', 'sumby_hispanbyP12HAI', 'prob_hispanbyP12HAI', 'hispanbyP5', 'preccount_hispanbyP5', 'sumby_hispanbyP5', 'prob_hispanbyP5', 'hispanbyP12HAI_counter', 'hispanbyP5_co

c:\Users\nathanael99\MyProjects\GitHub\intersect-community-data\pyncoda\CommunitySourceData\api_census_gov\acg_02a_add_categorical_char.py:788: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  set_flag_df.loc[conditions, self.flaggeo_var] = self.round+.5


After update observations geovar flag set 1 = 37039
    Check random merge results for primary data.
Check by geovar flag randagePCT12_Tract2010_flagsetrm
Observations flag not equal to 0 37039
Input and output data have the same length 37039
Outputdata has 37039 Observations with predicted randagePCT12
Percent left to predict:  0.00
    Overwrite input data with update output data.
Flag vars available ['randagePCT12_flagsetrm', 'randagePCT12_Tract2010_flagsetrm']
Before update output_df observations flag set 1 = 0
Before update results observations flag set 1 = 37039
After update observations geovar flag set 1 = 37039
    Check random merge results for secondary data.
Check by geovar flag randagePCT12_Tract2010_flagsetrm
Observations flag not equal to 0 37039
Input and output data have the same length 37039
Outputdata has 37039 Observations with predicted randagePCT12
Percent left to predict:  0.00
    Overwrite input data with update output data.

++++++++++++++++++++++++++++++++++++

,precid,Tract2010,Block2010,Block2010str,sex,minageyrs,maxageyrs,race,hispan,prec_counter,...,prob_hispanbyP5,hispanbyP12HAI_counter,hispanbyP5_counter,preccount_hispanbyP12HAIupdated,preccount_hispanbyP5updated,randageP12,agegroupP12,randagePCT12,randagePCT12_flagsetrm,randagePCT12_Tract2010_flagsetrm
0,B410079501001001P001,41007950100,410079501001001,B410079501001001,1.0,21.0,21.0,1.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,21.0,7.0,21.0,1,1.0
1,B410079501001001P002,41007950100,410079501001001,B410079501001001,1.0,55.0,59.0,1.0,0.0,2.0,...,NaN,NaN,NaN,NaN,NaN,58.0,15.0,56.0,1,1.0
2,B410079501001001P003,41007950100,410079501001001,B410079501001001,1.0,67.0,69.0,1.0,0.0,3.0,...,NaN,NaN,NaN,NaN,NaN,67.0,19.0,69.0,1,1.0


In [17]:
# Check vintage, dataset, and structure of all 7 disability tables (B18101–B18107)
from pyncoda.CommunitySourceData.api_census_gov.acg_00h_disability_ACS5yr2012 import (
    disability_B18101_varstem_roots,
    disability_B18102_varstem_roots,
    disability_B18103_varstem_roots,
    disability_B18104_varstem_roots,
    disability_B18105_varstem_roots,
    disability_B18106_varstem_roots,
    disability_B18107_varstem_roots,
)

all_dicts = {
    'B18101 (overall disability)':       disability_B18101_varstem_roots,
    'B18102 (hearing difficulty)':       disability_B18102_varstem_roots,
    'B18103 (vision difficulty)':        disability_B18103_varstem_roots,
    'B18104 (cognitive difficulty)':     disability_B18104_varstem_roots,
    'B18105 (ambulatory difficulty)':    disability_B18105_varstem_roots,
    'B18106 (self-care difficulty)':     disability_B18106_varstem_roots,
    'B18107 (indep. living difficulty)': disability_B18107_varstem_roots,
}

for name, d in all_dicts.items():
    md = d['metadata']
    var_count = len(d[md['group'] + '_'])   # e.g. d['B18101_']
    print(f"{name}")
    print(f"  vintage      : {md['vintage']}")
    print(f"  dataset_name : {md['dataset_name']}")
    print(f"  new_char     : {md['new_char']}")
    print(f"  char_vars    : {md['char_vars']}")
    print(f"  variables    : {var_count} variable codes")
    print()

B18101 (overall disability)
  vintage      : 2012
  dataset_name : acs/acs5
  new_char     : disability
  char_vars    : ['sex', 'agegroupB18101', 'disability']
  variables    : 24 variable codes

B18102 (hearing difficulty)
  vintage      : 2012
  dataset_name : acs/acs5
  new_char     : hearing_difficulty
  char_vars    : ['sex', 'agegroupB18101', 'hearing_difficulty']
  variables    : 24 variable codes

B18103 (vision difficulty)
  vintage      : 2012
  dataset_name : acs/acs5
  new_char     : vision_difficulty
  char_vars    : ['sex', 'agegroupB18101', 'vision_difficulty']
  variables    : 24 variable codes

B18104 (cognitive difficulty)
  vintage      : 2012
  dataset_name : acs/acs5
  new_char     : cognitive_difficulty
  char_vars    : ['sex', 'agegroupB18101', 'cognitive_difficulty']
  variables    : 20 variable codes

B18105 (ambulatory difficulty)
  vintage      : 2012
  dataset_name : acs/acs5
  new_char     : ambulatory_difficulty
  char_vars    : ['sex', 'agegroupB18101', 

In [18]:
import requests

# Test the exact URL the API is calling
url = ("https://api.census.gov/data/2012/acs/acs5"
       "?get=GEO_ID,B18101_004E,B18101_005E"
       "&in=state:48&in=county:167&for=tract:*")

# Also try without underscore (as the code is using)
url2 = ("https://api.census.gov/data/2012/acs/acs5"
        "?get=GEO_ID,B18101004E,B18101005E"
        "&in=state:48&in=county:167&for=tract:*")

r1 = requests.get(url)
print(f"URL with underscore - Status: {r1.status_code}")
print(r1.text[:300])

print()

r2 = requests.get(url2)
print(f"URL without underscore - Status: {r2.status_code}")
print(r2.text[:300])

URL with underscore - Status: 200
<html style="font-size: 14px;">

<head>
    <title>Missing Key</title>
    <link rel="icon" type="image/x-icon" href="favicon.ico">
    <link rel="stylesheet" type="text/css" href="assets/styles.css">
    <script type="text/javascript" src="assets/jquery-1.4.4.min.js"></script>
    <script type="tex

URL without underscore - Status: 200
<html style="font-size: 14px;">

<head>
    <title>Missing Key</title>
    <link rel="icon" type="image/x-icon" href="favicon.ico">
    <link rel="stylesheet" type="text/css" href="assets/styles.css">
    <script type="text/javascript" src="assets/jquery-1.4.4.min.js"></script>
    <script type="tex


In [19]:
# ==============================================================
# Step 6b: Fetch tract-level disability data (B18101–B18107)
# ==============================================================
from pyncoda.CommunitySourceData.api_census_gov.acg_01a_BaseInventory import BaseInventory
from pyncoda.CommunitySourceData.api_census_gov.acg_00h_disability_ACS5yr2012 import (
    disability_B18101_varstem_roots,
    disability_B18102_varstem_roots,
    disability_B18103_varstem_roots,
    disability_B18104_varstem_roots,
    disability_B18105_varstem_roots,
    disability_B18106_varstem_roots,
    disability_B18107_varstem_roots,
)

# All 7 disability tables to fetch
disability_tables = {
    'B18101': disability_B18101_varstem_roots,   # Overall disability
    'B18102': disability_B18102_varstem_roots,   # Hearing difficulty
    'B18103': disability_B18103_varstem_roots,   # Vision difficulty
    'B18104': disability_B18104_varstem_roots,   # Cognitive difficulty
    'B18105': disability_B18105_varstem_roots,   # Ambulatory difficulty
    'B18106': disability_B18106_varstem_roots,   # Self-care difficulty
    'B18107': disability_B18107_varstem_roots,   # Independent living difficulty
}

# Fetch each table from Census API
for group, varstem_dict in disability_tables.items():
    print(f"\n--- Fetching {group} ---")
    tract_df[group] = BaseInventory.get_apidata(
        state_county  = state_county,
        geo_level     = 'tract',
        vintage       = '2012',
        mutually_exclusive_varstems_roots_dictionaries = [varstem_dict],
        outputfolders = prec_outputfolders,
        outputfile    = group
    )
    print(f"{group} fetched. Shape: {tract_df[group].shape}")
    print(f"Columns: {list(tract_df[group].columns)}")
    display(tract_df[group].head(3))

print(f"\n✅ All 7 disability tables fetched successfully")
print(f"Tables in tract_df: {[k for k in tract_df.keys() if k.startswith('B181')]}")


--- Fetching B18101 ---
{'top': 'OutputData/Seaside_OR_NSI', 'logfiles': 'OutputData/Seaside_OR_NSI/00_logfiles', 'CommunitySourceData': 'OutputData/Seaside_OR_NSI/01_CommunitySourceData', 'TidyCommunitySourceData': 'OutputData/Seaside_OR_NSI/02_TidyCommunitySourceData', 'BaseInventory': 'OutputData/Seaside_OR_NSI/03_BaseInventory', 'RandomMerge': 'OutputData/Seaside_OR_NSI/04_RandomMerge'}
File OutputData/Seaside_OR_NSI/02_TidyCommunitySourceData/B18101_41007_2012.csv Already exists - Skipping API Call.
B18101 fetched. Shape: (36381, 9)
Columns: ['index', 'GEO_ID', 'state', 'county', 'tract', 'sex', 'agegroupB18101', 'disability', 'prec_counter']


,index,GEO_ID,state,county,tract,sex,agegroupB18101,disability,prec_counter
0,19,1400000US41007950100,41,7,950100,1,1,0,1
1,19,1400000US41007950100,41,7,950100,1,1,0,2
2,19,1400000US41007950100,41,7,950100,1,1,0,3



--- Fetching B18102 ---
{'top': 'OutputData/Seaside_OR_NSI', 'logfiles': 'OutputData/Seaside_OR_NSI/00_logfiles', 'CommunitySourceData': 'OutputData/Seaside_OR_NSI/01_CommunitySourceData', 'TidyCommunitySourceData': 'OutputData/Seaside_OR_NSI/02_TidyCommunitySourceData', 'BaseInventory': 'OutputData/Seaside_OR_NSI/03_BaseInventory', 'RandomMerge': 'OutputData/Seaside_OR_NSI/04_RandomMerge'}
['sex', 'agegroupB18101', 'hearing_difficulty']

**********************************
Obtain data from Census API SEX BY AGE BY HEARING DIFFICULTY
    Obtaining data for B18102_ SEX BY AGE BY HEARING DIFFICULTY by Race and Hispanic Not Applicable
       Census API data from: https://api.census.gov/data/2012/acs/acs5?get=GEO_ID,B18102_004E,B18102_005E,B18102_007E,B18102_008E,B18102_010E,B18102_011E,B18102_013E,B18102_014E,B18102_016E,B18102_017E,B18102_019E,B18102_020E,B18102_023E,B18102_024E,B18102_026E,B18102_027E,B18102_029E,B18102_030E,B18102_032E,B18102_033E,B18102_035E,B18102_036E,B18102_038E,B1

,index,GEO_ID,state,county,tract,sex,agegroupB18101,hearing_difficulty,prec_counter
0,19,1400000US41007950100,41,007,950100,1,1,0,1
1,19,1400000US41007950100,41,007,950100,1,1,0,2
2,19,1400000US41007950100,41,007,950100,1,1,0,3



--- Fetching B18103 ---
{'top': 'OutputData/Seaside_OR_NSI', 'logfiles': 'OutputData/Seaside_OR_NSI/00_logfiles', 'CommunitySourceData': 'OutputData/Seaside_OR_NSI/01_CommunitySourceData', 'TidyCommunitySourceData': 'OutputData/Seaside_OR_NSI/02_TidyCommunitySourceData', 'BaseInventory': 'OutputData/Seaside_OR_NSI/03_BaseInventory', 'RandomMerge': 'OutputData/Seaside_OR_NSI/04_RandomMerge'}
['sex', 'agegroupB18101', 'vision_difficulty']

**********************************
Obtain data from Census API SEX BY AGE BY VISION DIFFICULTY
    Obtaining data for B18103_ SEX BY AGE BY VISION DIFFICULTY by Race and Hispanic Not Applicable
       Census API data from: https://api.census.gov/data/2012/acs/acs5?get=GEO_ID,B18103_004E,B18103_005E,B18103_007E,B18103_008E,B18103_010E,B18103_011E,B18103_013E,B18103_014E,B18103_016E,B18103_017E,B18103_019E,B18103_020E,B18103_023E,B18103_024E,B18103_026E,B18103_027E,B18103_029E,B18103_030E,B18103_032E,B18103_033E,B18103_035E,B18103_036E,B18103_038E,B1810

...


--- Fetching B18106 ---
{'top': 'OutputData/Seaside_OR_NSI', 'logfiles': 'OutputData/Seaside_OR_NSI/00_logfiles', 'CommunitySourceData': 'OutputData/Seaside_OR_NSI/01_CommunitySourceData', 'TidyCommunitySourceData': 'OutputData/Seaside_OR_NSI/02_TidyCommunitySourceData', 'BaseInventory': 'OutputData/Seaside_OR_NSI/03_BaseInventory', 'RandomMerge': 'OutputData/Seaside_OR_NSI/04_RandomMerge'}
['sex', 'agegroupB18101', 'selfcare_difficulty']

**********************************
Obtain data from Census API SEX BY AGE BY SELF-CARE DIFFICULTY
    Obtaining data for B18106_ SEX BY AGE BY SELF-CARE DIFFICULTY by Race and Hispanic Not Applicable
       Census API data from: https://api.census.gov/data/2012/acs/acs5?get=GEO_ID,B18106_004E,B18106_005E,B18106_007E,B18106_008E,B18106_010E,B18106_011E,B18106_013E,B18106_014E,B18106_016E,B18106_017E,B18106_020E,B18106_021E,B18106_023E,B18106_024E,B18106_026E,B18106_027E,B18106_029E,B18106_030E,B18106_032E,B18106_033E&in=state:41&in=county:007&for=tra

,index,GEO_ID,state,county,tract,sex,agegroupB18101,selfcare_difficulty,prec_counter
0,19,1400000US41007950100,41,007,950100,1,2,0,1
1,19,1400000US41007950100,41,007,950100,1,2,0,2
2,19,1400000US41007950100,41,007,950100,1,2,0,3



--- Fetching B18107 ---
{'top': 'OutputData/Seaside_OR_NSI', 'logfiles': 'OutputData/Seaside_OR_NSI/00_logfiles', 'CommunitySourceData': 'OutputData/Seaside_OR_NSI/01_CommunitySourceData', 'TidyCommunitySourceData': 'OutputData/Seaside_OR_NSI/02_TidyCommunitySourceData', 'BaseInventory': 'OutputData/Seaside_OR_NSI/03_BaseInventory', 'RandomMerge': 'OutputData/Seaside_OR_NSI/04_RandomMerge'}
['sex', 'agegroupB18101', 'indliving_difficulty']

**********************************
Obtain data from Census API SEX BY AGE BY INDEPENDENT LIVING DIFFICULTY
    Obtaining data for B18107_ SEX BY AGE BY INDEPENDENT LIVING DIFFICULTY by Race and Hispanic Not Applicable
       Census API data from: https://api.census.gov/data/2012/acs/acs5?get=GEO_ID,B18107_004E,B18107_005E,B18107_007E,B18107_008E,B18107_010E,B18107_011E,B18107_013E,B18107_014E,B18107_017E,B18107_018E,B18107_020E,B18107_021E,B18107_023E,B18107_024E,B18107_026E,B18107_027E&in=state:41&in=county:007&for=tract:*

***********************

,index,GEO_ID,state,county,tract,sex,agegroupB18101,indliving_difficulty,prec_counter
0,7,1400000US41007950100,41,007,950100,1,3,1,1
1,7,1400000US41007950100,41,007,950100,1,3,1,2
2,7,1400000US41007950100,41,007,950100,1,3,1,3



✅ All 7 disability tables fetched successfully
Tables in tract_df: ['B18101', 'B18102', 'B18103', 'B18104', 'B18105', 'B18106', 'B18107']


In [20]:
from pyncoda.CommunitySourceData.api_census_gov.acg_02c_agefunctions import (
    add_randage,
    add_P12age_groups,
    add_B18101age_groups
)

In [21]:
# ==============================================================
# Step 7: Add B18101 age groups to person records (used by all 7 disability tables)
# ==============================================================
prec_age_df['primary'] = add_B18101age_groups(
    prec_age_df['primary'],
    varname = 'randagePCT12'
)
print(f"Step 7 complete!")

print(f"\nPerson records agegroupB18101 value counts:")
print(prec_age_df['primary']['agegroupB18101'].value_counts().sort_index())

# Loop through all 7 disability tables for diagnostics
disability_char_cols = {
    'B18101': 'disability',
    'B18102': 'hearing_difficulty',
    'B18103': 'vision_difficulty',
    'B18104': 'cognitive_difficulty',
    'B18105': 'ambulatory_difficulty',
    'B18106': 'selfcare_difficulty',
    'B18107': 'indliving_difficulty',
}

for table, char_col in disability_char_cols.items():
    print(f"\n--- {table} ---")
    print(f"  Shape: {tract_df[table].shape}")
    print(f"  {char_col} value counts:")
    print(tract_df[table][char_col].value_counts().sort_index())
    print(f"  agegroupB18101 value counts:")
    print(tract_df[table]['agegroupB18101'].value_counts().sort_index())

print(f"\nPerson records shape: {prec_age_df['primary'].shape}")

Step 7 complete!

Person records agegroupB18101 value counts:
agegroupB18101
1.0     2076
2.0     5525
3.0     7540
4.0    15738
5.0     3470
6.0     2690
Name: count, dtype: int64

--- B18101 ---
  Shape: (36381, 9)
  disability value counts:
disability
0    29934
1     6447
Name: count, dtype: int64
  agegroupB18101 value counts:
agegroupB18101
1     1971
2     5584
3     7109
4    15465
5     3606
6     2646
Name: count, dtype: int64

--- B18102 ---
  Shape: (36381, 9)
  hearing_difficulty value counts:
hearing_difficulty
0    34623
1     1758
Name: count, dtype: int64
  agegroupB18101 value counts:
agegroupB18101
1     1971
2     5584
3     7109
4    15465
5     3606
6     2646
Name: count, dtype: int64

--- B18103 ---
  Shape: (36381, 9)
  vision_difficulty value counts:
vision_difficulty
0    35430
1      951
Name: count, dtype: int64
  agegroupB18101 value counts:
agegroupB18101
1     1971
2     5584
3     7109
4    15465
5     3606
6     2646
Name: count, dtype: int64

--- B181

In [22]:
# =============================================================
# Step 8: Random Merge ALL 7 Disability Types — UNIVERSE-SAFE
#   Phase A: Generate ONE file per disability type (7 separate files)
#   Phase B: Combine all 7 into a single final file
#
# Universe restrictions enforced via pre-filtering:
#   B18101/B18102/B18103: all ages
#   B18104/B18105/B18106: 5+ years   (agegroupB18101 >= 2)
#   B18107:               18+ years  (agegroupB18101 >= 3)
#
# 4-round merge strategy (within universe only):
#   Round 1: Tract + Sex + AgegroupB18101  (most precise)
#   Round 2: Tract + Sex                   (drop age group)
#   Round 3: Tract only                    (drop sex)
#   Round 4: County only                   (broadest fallback)
# =============================================================
import os
import pandas as pd
from pyncoda.CommunitySourceData.api_census_gov.acg_02a_add_categorical_char import add_new_char_by_random_merge_2dfs

# Map each Census table to (column_name, minimum_eligible_agegroup)
# agegroupB18101: 1=Under5, 2=5-17, 3=18-34, 4=35-64, 5=65-74, 6=75+
disability_tables = {
    'B18101': ('disability',             1),   # all ages
    'B18102': ('hearing_difficulty',     1),   # all ages
    'B18103': ('vision_difficulty',      1),   # all ages
    'B18104': ('cognitive_difficulty',   2),   # 5+ years
    'B18105': ('ambulatory_difficulty',  2),   # 5+ years
    'B18106': ('selfcare_difficulty',    2),   # 5+ years
    'B18107': ('indliving_difficulty',   3),   # 18+ years
}

# Define the 4-round merge strategy (same for all 7 tables)
rounds = {
    'options': {
        'option1': {
            'notes'            : 'Round 1: Tract + Sex + AgegroupB18101 (most precise)',
            'common_group_vars': ['agegroupB18101'],
            'by_groups'        : {'All': {'by_variables': ['sex']}}
        },
        'option2': {
            'notes'            : 'Round 2: Tract + Sex (drop age group)',
            'common_group_vars': [],
            'by_groups'        : {'All': {'by_variables': ['sex']}}
        },
        'option3': {
            'notes'            : 'Round 3: Tract only (drop sex)',
            'common_group_vars': [],
            'by_groups'        : {'All': {'by_variables': []}}
        },
        'option4': {
            'notes'            : 'Round 4: County level (broadest fallback)',
            'common_group_vars': [],
            'by_groups'        : {'All': {'by_variables': []}}
        }
    },
    'geo_levels': ['Tract', 'Tract', 'Tract', 'County']
}

# =============================================================
# PHASE A — Generate one standalone file per disability type
# =============================================================
print("\n" + "="*70)
print("PHASE A: Generating one file per disability type")
print("="*70)

# Base person records (have agegroupB18101 from Step 7) — used as input for each merge
base_df = prec_age_df['primary'].copy()

# Container to hold the result of each disability merge (one entry per table)
disability_results = {}

for table, (char_col, min_agegroup) in disability_tables.items():
    print(f"\n{'─'*70}")
    print(f"[{table}] Merging → '{char_col}'  (universe: agegroup >= {min_agegroup})")
    print(f"{'─'*70}")
    
    # Filter base df to ONLY in-universe records before merging
    eligible_mask = base_df['agegroupB18101'] >= min_agegroup
    eligible_df   = base_df[eligible_mask].copy()
    out_count     = (~eligible_mask).sum()
    
    print(f"  In-universe records:     {len(eligible_df):,}")
    print(f"  Out-of-universe records: {out_count:,} (will be set to -999)")
    
    # 4-round random merge — only on eligible records, no leakage possible
    add_char = add_new_char_by_random_merge_2dfs(
        dfs = {
            'primary':   {'data'      : eligible_df,
                          'primarykey': 'precid',
                          'geolevel'  : 'Block',
                          'geovintage': '2010',
                          'notes'     : f'Eligible person-level data ({char_col})'},
            'secondary': {'data'      : tract_df[table],
                          'primarykey': f'uniqueid{table}',
                          'geolevel'  : 'Tract',
                          'geovintage': '2010',
                          'notes'     : f'Tract-level {table} counts by sex and age'}
        },
        seed              = seed,
        common_group_vars = ['agegroupB18101'],
        new_char          = char_col,
        geolevel          = 'Tract',
        geovintage        = '2010',
        by_groups         = {'All': {'by_variables': ['sex']}},
        fillna_value      = -999,
        state_county      = state_county,
        outputfile        = f'prec_{char_col}',
        outputfolder      = prec_outputfolders['RandomMerge']
    )
    
    merged          = add_char.run_random_merge_2dfs(rounds)
    eligible_result = merged['primary']
    
    # Build the standalone file for THIS disability type:
    #   Start from the FULL base population
    #   Bring in disability_col + flag columns from the eligible result
    #   Out-of-universe records get -999
    new_cols  = [c for c in eligible_result.columns if c not in base_df.columns]
    keep_cols = ['precid'] + new_cols
    
    standalone_df = (
        base_df.set_index('precid')
               .join(eligible_result[keep_cols].set_index('precid'), how='left')
               .reset_index()
    )
    
    # Fill out-of-universe records with -999 for char column AND flag columns
    for col in new_cols:
        standalone_df[col] = standalone_df[col].fillna(-999)
    
    # Save the standalone file for this disability type
    out_path = os.path.join(
        prec_outputfolders['RandomMerge'],
        f'prec_{char_col}_{state_county}_2010_rs{seed}.csv'
    )
    standalone_df.to_csv(out_path, index=False)
    print(f"  Saved → {out_path}")
    
    # Quick verification: universe enforcement
    out_violations = ((standalone_df['agegroupB18101'] < min_agegroup) &
                      (standalone_df[char_col].isin([0, 1]))).sum()
    in_universe   = standalone_df[standalone_df['agegroupB18101'] >= min_agegroup]
    in_assigned   = in_universe[char_col].isin([0, 1]).sum()
    in_with       = (in_universe[char_col] == 1).sum()
    in_rate       = in_assigned / len(in_universe) * 100 if len(in_universe) else 0
    with_rate     = in_with / in_assigned * 100 if in_assigned else 0
    
    status = "✅" if out_violations == 0 else "❌"
    print(f"  {status} Universe violations: {out_violations}")
    print(f"     Within-universe assignment rate: {in_rate:.2f}%")
    print(f"     With-disability rate (in universe): {with_rate:.2f}%")
    print(f"     Value counts:")
    print(standalone_df[char_col].value_counts().sort_index().to_string().replace('\n', '\n     '))
    
    # Stash for Phase B
    disability_results[char_col] = standalone_df[['precid', char_col] + 
                                                  [c for c in new_cols if c != char_col]]

print(f"\n{'='*70}")
print(f"✅ PHASE A COMPLETE: 7 standalone disability files generated")
print(f"{'='*70}")

# =============================================================
# PHASE B — Combine all 7 disability columns into ONE final file
# =============================================================
print("\n" + "="*70)
print("PHASE B: Combining all 7 disability types into one final file")
print("="*70)

# Start with base person records
output_df = base_df.copy()

# Merge each disability result onto the base — only pull the columns
# specifically for that disability type (avoids auxiliary column collisions)
for char_col, sub_df in disability_results.items():
    # Keep only: precid + the char column + columns that explicitly start with char_col_
    # (these are the disability-specific flag columns like 'disability_flagsetrm')
    keep_cols = ['precid'] + [
        c for c in sub_df.columns
        if c == char_col or c.startswith(f'{char_col}_')
    ]
    sub_df_clean = sub_df[keep_cols]
    
    output_df = output_df.merge(sub_df_clean, on='precid', how='left')
    
    # Make sure -999 stays as -999 (not NaN) for the columns we just added
    for col in sub_df_clean.columns:
        if col != 'precid' and col in output_df.columns:
            output_df[col] = output_df[col].fillna(-999)

# Save the combined file
combined_path = os.path.join(
    prec_outputfolders['RandomMerge'],
    f'prec_disability_all_{state_county}_2010_rs{seed}.csv'
)
output_df.to_csv(combined_path, index=False)
print(f"\nFinal combined file saved → {combined_path}")
print(f"Shape: {output_df.shape}")

# Final summary table — universe-aware
print(f"\n{'─'*80}")
print(f"{'Disability Type':<25} {'Universe':<10} {'In-Univ Assigned %':>20} {'With (1) %':>12}")
print(f"{'─'*80}")
universe_labels = {1: 'all ages', 2: '5+ yrs', 3: '18+ yrs'}
for table, (char_col, min_ag) in disability_tables.items():
    in_universe = output_df[output_df['agegroupB18101'] >= min_ag]
    universe_size = len(in_universe)
    assigned = in_universe[char_col].isin([0, 1]).sum()
    assigned_pct = assigned / universe_size * 100 if universe_size else 0
    with_pct = ((in_universe[char_col] == 1).sum() / assigned * 100) if assigned else 0
    print(f"{char_col:<25} {universe_labels[min_ag]:<10} {assigned_pct:>19.2f}% {with_pct:>11.2f}%")

# Verify universe enforcement is clean across all 7 columns
print(f"\n{'─'*80}")
print("Universe enforcement verification:")
print(f"{'─'*80}")
all_clean = True
for table, (char_col, min_ag) in disability_tables.items():
    out_violations = ((output_df['agegroupB18101'] < min_ag) &
                      (output_df[char_col].isin([0, 1]))).sum()
    status = "✅" if out_violations == 0 else "❌"
    if out_violations > 0:
        all_clean = False
    print(f"{status} {char_col:<25}: {out_violations} out-of-universe assignments")

print()
if all_clean:
    print("✅ ALL 7 DISABILITY TYPES PASS UNIVERSE ENFORCEMENT")
else:
    print("❌ SOME DISABILITY TYPES HAVE UNIVERSE VIOLATIONS - REVIEW REQUIRED")

# Show sample
display_cols = ['precid', 'sex', 'agegroupB18101', 'randagePCT12'] + \
               [c for t, (c, m) in disability_tables.items()]
display(output_df[display_cols].head(5))


PHASE A: Generating one file per disability type

──────────────────────────────────────────────────────────────────────
[B18101] Merging → 'disability'  (universe: agegroup >= 1)
──────────────────────────────────────────────────────────────────────
  In-universe records:     37,039
  Out-of-universe records: 0 (will be set to -999)
Round 1
Performing random merge at geography level: Tract
Round 1: Tract + Sex + AgegroupB18101 (most precise)
Running random merge by ['Tract2010', 'sex', 'agegroupB18101']
    Setting up  primary data with primary key and flags
Geolevels available []
Geolvarids available ['Tract2010', 'Block2010']
Adding Tract2010 expected length 11
Dataframe has Block 2010 for new geovar Tract2010
Confirming Tract2010 has expected length.
Checking primary key name precid
Primary key variable precid is unique.
Primary key precid has no missing values
Initializing primary flag set variable for disability
New flag: disability_flagsetrm
['precid', 'Tract2010', 'Block2010',

c:\Users\nathanael99\MyProjects\GitHub\intersect-community-data\pyncoda\CommunitySourceData\api_census_gov\acg_02a_add_categorical_char.py:765: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  set_flag_df.loc[conditions, self.flaggeo_var] = .5


Geolevels available ['state', 'tract']
Geolvarids available ['Tract2010']
Adding County2010 expected length 5
Dataframe has Tract 2010 for new geovar County2010
Confirming County2010 has expected length.
Checking primary key name uniqueidB18101
Primary key variable uniqueidB18101 is unique.
Primary key uniqueidB18101 has no missing values
['uniqueidB18101', 'County2010', 'Tract2010', 'index', 'GEO_ID', 'state', 'county', 'tract', 'sex', 'agegroupB18101', 'prec_counter', 'disability', 'randagePCT12_flagsetrm', 'randagePCT12_Tract2010_flagsetrm', 'disability_flagsetrm', 'disability_Tract2010_flagsetrm']
Initializing geovar flag set variable for disability at geolevel County2010
New flag: disability_County2010_flagsetrm
Observations without primary flag set 584
Observations without geovar flag set 36381
After updated observations without geovar flag set 584
Setting 584 flags for disability secondary data not used.
0 observations do not have required variable County2010
0 observations do n

c:\Users\nathanael99\MyProjects\GitHub\intersect-community-data\pyncoda\CommunitySourceData\api_census_gov\acg_02a_add_categorical_char.py:765: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  set_flag_df.loc[conditions, self.flaggeo_var] = .5


Geolevels available ['state', 'county', 'tract']
Geolvarids available ['Tract2010']
Adding County2010 expected length 5
Dataframe has Tract 2010 for new geovar County2010
Confirming County2010 has expected length.
Checking primary key name uniqueidB18102
Primary key variable uniqueidB18102 is unique.
Primary key uniqueidB18102 has no missing values
['uniqueidB18102', 'County2010', 'Tract2010', 'index', 'GEO_ID', 'state', 'county', 'tract', 'sex', 'agegroupB18101', 'prec_counter', 'hearing_difficulty', 'randagePCT12_flagsetrm', 'randagePCT12_Tract2010_flagsetrm', 'hearing_difficulty_flagsetrm', 'hearing_difficulty_Tract2010_flagsetrm']
Initializing geovar flag set variable for hearing_difficulty at geolevel County2010
New flag: hearing_difficulty_County2010_flagsetrm
Observations without primary flag set 584
Observations without geovar flag set 36381
After updated observations without geovar flag set 584
Setting 584 flags for hearing_difficulty secondary data not used.
0 observations do

...

c:\Users\nathanael99\MyProjects\GitHub\intersect-community-data\pyncoda\CommunitySourceData\api_census_gov\acg_02a_add_categorical_char.py:765: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  set_flag_df.loc[conditions, self.flaggeo_var] = .5


Primary data frame has extra selfcare_difficulty  observations with no match: 693
Observations with no match filled with -999
Merge found extra selfcare_difficulty  observations: 140
    Set Flags after Merge
['precid', 'County2010', 'sex', 'agegroupB18101', 'random_mergeorder', 'uniqueidB18106', 'selfcare_difficulty', 'check_merge', 'randagePCT12_flagsetrm', 'randagePCT12_Tract2010_flagsetrm', 'selfcare_difficulty_flagsetrm', 'selfcare_difficulty_Tract2010_flagsetrm', 'selfcare_difficulty_County2010_flagsetrm']
Observations without primary flag set 1287
Observations without geovar flag set 1287
After updated observations without geovar flag set 1287
Round = 13
Setting 454 flags for observations set by random merge using both primary and secondary data.
Setting 0 flags for observations set by random merge using only primary data.
0 observations do not have required variable County2010
0 observations do not have required variable sex
0 observations do not have required variable agegroup

c:\Users\nathanael99\MyProjects\GitHub\intersect-community-data\pyncoda\CommunitySourceData\api_census_gov\acg_02a_add_categorical_char.py:765: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  set_flag_df.loc[conditions, self.flaggeo_var] = .5


Geolevels available ['state', 'county', 'tract']
Geolvarids available ['Tract2010']
Adding County2010 expected length 5
Dataframe has Tract 2010 for new geovar County2010
Confirming County2010 has expected length.
Checking primary key name uniqueidB18107
Primary key variable uniqueidB18107 is unique.
Primary key uniqueidB18107 has no missing values
['uniqueidB18107', 'County2010', 'Tract2010', 'index', 'GEO_ID', 'state', 'county', 'tract', 'sex', 'agegroupB18101', 'prec_counter', 'indliving_difficulty', 'randagePCT12_flagsetrm', 'randagePCT12_Tract2010_flagsetrm', 'indliving_difficulty_flagsetrm', 'indliving_difficulty_Tract2010_flagsetrm']
Initializing geovar flag set variable for indliving_difficulty at geolevel County2010
New flag: indliving_difficulty_County2010_flagsetrm
Observations without primary flag set 844
Observations without geovar flag set 28826
After updated observations without geovar flag set 844
Setting 844 flags for indliving_difficulty secondary data not used.
0 obs

,precid,sex,agegroupB18101,randagePCT12,disability,hearing_difficulty,vision_difficulty,cognitive_difficulty,ambulatory_difficulty,selfcare_difficulty,indliving_difficulty
0,B410079501001001P001,1.0,3.0,21.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,B410079501001001P002,1.0,4.0,56.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,B410079501001001P003,1.0,5.0,69.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,B410079501001001P004,1.0,6.0,75.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
4,B410079501001001P005,2.0,5.0,66.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Validate Disability Results

In [23]:
output_df.head()

,precid,Tract2010,Block2010,Block2010str,sex,minageyrs,maxageyrs,race,hispan,prec_counter,...,ambulatory_difficulty_Tract2010_flagsetrm,ambulatory_difficulty_County2010_flagsetrm,selfcare_difficulty,selfcare_difficulty_flagsetrm,selfcare_difficulty_Tract2010_flagsetrm,selfcare_difficulty_County2010_flagsetrm,indliving_difficulty,indliving_difficulty_flagsetrm,indliving_difficulty_Tract2010_flagsetrm,indliving_difficulty_County2010_flagsetrm
0,B410079501001001P001,41007950100,410079501001001,B410079501001001,1.0,21.0,21.0,1.0,0.0,1.0,...,1.0,-777.0,0.0,1.0,1.0,-777.0,0.0,1.0,1.0,-777.0
1,B410079501001001P002,41007950100,410079501001001,B410079501001001,1.0,55.0,59.0,1.0,0.0,2.0,...,1.0,-777.0,0.0,1.0,1.0,-777.0,0.0,1.0,1.0,-777.0
2,B410079501001001P003,41007950100,410079501001001,B410079501001001,1.0,67.0,69.0,1.0,0.0,3.0,...,1.0,-777.0,0.0,1.0,1.0,-777.0,0.0,1.0,1.0,-777.0
3,B410079501001001P004,41007950100,410079501001001,B410079501001001,1.0,75.0,79.0,1.0,0.0,4.0,...,1.0,-777.0,0.0,1.0,1.0,-777.0,0.0,1.0,1.0,-777.0
4,B410079501001001P005,41007950100,410079501001001,B410079501001001,2.0,65.0,66.0,1.0,0.0,5.0,...,1.0,-777.0,0.0,1.0,1.0,-777.0,0.0,1.0,1.0,-777.0


In [24]:
output_df[display_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
sex,37039.0,1.503037,0.499998,1.0,1.0,2.0,2.0,2.0
agegroupB18101,37039.0,3.568887,1.225990,1.0,3.0,4.0,4.0,6.0
randagePCT12,37039.0,41.012770,23.252288,0.0,21.0,43.0,59.0,103.0
disability,37039.0,-17.573234,131.990045,-999.0,0.0,0.0,0.0,1.0
hearing_difficulty,37039.0,-17.699830,131.972648,-999.0,0.0,0.0,0.0,1.0
vision_difficulty,37039.0,-17.721618,131.969641,-999.0,0.0,0.0,0.0,1.0
cognitive_difficulty,37039.0,-70.833473,256.557638,-999.0,0.0,0.0,0.0,1.0
ambulatory_difficulty,37039.0,-70.817679,256.562029,-999.0,0.0,0.0,0.0,1.0
selfcare_difficulty,37039.0,-70.870380,256.547373,-999.0,0.0,0.0,0.0,1.0
indliving_difficulty,37039.0,-221.459192,415.037982,-999.0,0.0,0.0,0.0,1.0


In [25]:
validate_disability_df = output_df[display_cols].copy()
validate_disability_df.head()

,precid,sex,agegroupB18101,randagePCT12,disability,hearing_difficulty,vision_difficulty,cognitive_difficulty,ambulatory_difficulty,selfcare_difficulty,indliving_difficulty
0,B410079501001001P001,1.0,3.0,21.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,B410079501001001P002,1.0,4.0,56.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,B410079501001001P003,1.0,5.0,69.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,B410079501001001P004,1.0,6.0,75.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
4,B410079501001001P005,2.0,5.0,66.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
# replace -999 with NaN for validation
validate_disability_df.replace(-999, pd.NA, inplace=True)
# convert all columns to appropriate types for validation
validate_disability_df = validate_disability_df.astype({
    'precid': 'string',
    'sex' : 'Int64',
    'agegroupB18101': 'Int64',
    'randagePCT12': 'Int64',
    'disability': 'Int64',
    'hearing_difficulty': 'Int64',
    'vision_difficulty': 'Int64',  
    'cognitive_difficulty': 'Int64',
    'ambulatory_difficulty': 'Int64',
    'selfcare_difficulty': 'Int64',
    'indliving_difficulty': 'Int64'
})
validate_disability_df.head()


,precid,sex,agegroupB18101,randagePCT12,disability,hearing_difficulty,vision_difficulty,cognitive_difficulty,ambulatory_difficulty,selfcare_difficulty,indliving_difficulty
0,B410079501001001P001,1,3,21,0,0,0,0,0,0,0
1,B410079501001001P002,1,4,56,0,0,0,0,0,0,0
2,B410079501001001P003,1,5,69,1,0,0,0,0,0,0
3,B410079501001001P004,1,6,75,1,1,1,0,0,0,0
4,B410079501001001P005,2,5,66,0,0,0,0,0,0,0


In [27]:
validate_disability_df.describe().T

,count,mean,std,min,25%,50%,75%,max
sex,37039.0,1.503037,0.499998,1.0,1.0,2.0,2.0,2.0
agegroupB18101,37039.0,3.568887,1.22599,1.0,3.0,4.0,4.0,6.0
randagePCT12,37039.0,41.01277,23.252288,0.0,21.0,43.0,59.0,103.0
disability,36381.0,0.177208,0.38185,0.0,0.0,0.0,0.0,1.0
hearing_difficulty,36381.0,0.048322,0.214449,0.0,0.0,0.0,0.0,1.0
vision_difficulty,36381.0,0.02614,0.159554,0.0,0.0,0.0,0.0,1.0
cognitive_difficulty,34410.0,0.0805,0.272069,0.0,0.0,0.0,0.0,1.0
ambulatory_difficulty,34410.0,0.097501,0.296643,0.0,0.0,0.0,0.0,1.0
selfcare_difficulty,34410.0,0.040773,0.197767,0.0,0.0,0.0,0.0,1.0
indliving_difficulty,28826.0,0.074932,0.263287,0.0,0.0,0.0,0.0,1.0


In [28]:

print("Disability:")
print(f"All Households: https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=ACSDT5Y2012.B18101")

Disability:
All Households: https://data.census.gov/cedsci/table?g=050XX00US41007&tid=ACSDT5Y2012.B18101


In [29]:
# select only population older than 65
validate_disability_df_over65 = validate_disability_df[validate_disability_df['randagePCT12'] >= 65]
validate_disability_df_over65.describe().T

,count,mean,std,min,25%,50%,75%,max
sex,6160.0,1.539773,0.498456,1.0,1.0,2.0,2.0,2.0
agegroupB18101,6160.0,5.436688,0.496016,5.0,5.0,5.0,6.0,6.0
randagePCT12,6160.0,74.619805,7.726402,65.0,68.0,73.0,80.0,103.0
disability,6094.0,0.36216,0.480664,0.0,0.0,0.0,1.0,1.0
hearing_difficulty,6094.0,0.162455,0.368898,0.0,0.0,0.0,0.0,1.0
vision_difficulty,6094.0,0.061044,0.23943,0.0,0.0,0.0,0.0,1.0
cognitive_difficulty,6120.0,0.114052,0.317901,0.0,0.0,0.0,0.0,1.0
ambulatory_difficulty,6120.0,0.235131,0.424115,0.0,0.0,0.0,0.0,1.0
selfcare_difficulty,6120.0,0.104902,0.306452,0.0,0.0,0.0,0.0,1.0
indliving_difficulty,6145.0,0.142718,0.349813,0.0,0.0,0.0,0.0,1.0


### Compare the above to 
https://data.census.gov/table/ACSST5Y2012.S1810?q=Disability&g=050XX00US41007

Task - make a new pop table results that creates summary stats for disability by age groups. The numbers in the table should match the values on data.census.gov